In [2]:
from pathlib import Path
import pickle

import joblib


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# путь к старому проекту со спринта 2
OLD_PROJECT_ROOT = Path("/home/mle-user/mle-project-sprint-1-v001/part2_dvc")

DATA_PATH = OLD_PROJECT_ROOT / "data" / "initial_data.csv"
MODEL_PATH = OLD_PROJECT_ROOT / "models" / "fitted_model.pkl"

EXPERIMENT_NAME = "real_estate_price"
RUN_NAME = "baseline_train_valid_test"
REGISTRY_MODEL_NAME = "real_estate_price_model"

# mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_experiment(EXPERIMENT_NAME)


In [ ]:
sys.path.append("/home/mle-user/mle-project-sprint-1-v001/part2_dvc/scripts")



In [4]:
DATA_PATH

PosixPath('/home/mle-user/mle-project-sprint-1-v001/part2_dvc/data/initial_data.csv')

In [3]:
df = pd.read_csv(DATA_PATH)

TARGET_COL = "price"  

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]


X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.25,          
    random_state=42
)

X_train.shape, X_valid.shape, X_test.shape


((56091, 20), (18698, 20), (18698, 20))

In [10]:
pd.options.display.max_columns = 100
pd.options.display.max_rows = 64

In [11]:
df

,building_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,flat_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,floors_total_bin,floor_bin,rooms_bin,build_year_bin,price
0,6220,NaN,6,55.717113,37.781120,2.64,84,NaN,True,0,NaN,9.90,19.900000,NaN,False,35.099998,low,low,low,med,9500000.0
1,18012,NaN,2,55.794849,37.608013,3.00,97,NaN,True,1,NaN,0.00,16.600000,NaN,False,43.000000,low,low,low,hig,13500000.0
2,17821,NaN,4,55.740040,37.761742,2.70,80,NaN,True,2,NaN,9.00,32.000000,NaN,False,56.000000,low,low,low,hig,13500000.0
3,9293,NaN,1,55.808807,37.707306,2.60,208,NaN,True,4,NaN,3.00,14.000000,NaN,False,24.000000,low,low,low,med,5200000.0
4,23964,NaN,4,55.724728,37.743069,2.70,192,NaN,True,5,NaN,0.00,0.000000,NaN,False,51.009998,low,low,low,hig,8490104.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93482,9503,NaN,4,55.740402,37.834579,2.64,428,NaN,True,141356,NaN,6.00,42.000000,NaN,False,64.000000,low,low,low,med,10800000.0
93483,3162,NaN,1,55.727470,37.768677,2.48,80,NaN,False,141358,NaN,5.28,28.330000,NaN,False,41.110001,low,low,low,med,7400000.0
93484,6513,NaN,4,55.704315,37.506584,2.64,72,NaN,True,141359,NaN,5.30,20.000000,NaN,False,31.500000,low,low,low,med,9700000.0
93485,23952,NaN,4,55.699863,37.939564,2.70,480,NaN,True,141360,NaN,13.80,33.700001,NaN,False,65.300003,med,med,low,hig,11750000.0


In [8]:
df.columns

Index(['building_id', 'build_year', 'building_type_int', 'latitude',
       'longitude', 'ceiling_height', 'flats_count', 'floors_total',
       'has_elevator', 'flat_id', 'floor', 'kitchen_area', 'living_area',
       'rooms', 'is_apartment', 'total_area', 'floors_total_bin', 'floor_bin',
       'rooms_bin', 'build_year_bin', 'price'],
      dtype='object')

In [13]:
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Путь к исходным данным
# df = pd.read_csv(DATA_PATH)

# Базовая функция трансформации

def transform(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # Удаление дубликатов
    if "flat_id" in df.columns:
        df = df.drop_duplicates(subset=["flat_id"])
    else:
        df = df.drop_duplicates()

    # Приведение логических колонок к int
    bool_cols = df.select_dtypes(include=["bool"]).columns
    for col in bool_cols:
        df[col] = df[col].astype(int)

    # Заполнение пропусков
    num_cols = df.select_dtypes(include=["number"]).columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in num_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    for col in cat_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode().iloc[0])

    # Выбросы только для адекватных числовых признаков
    outlier_cols = [
        "price",
        "total_area",
        "living_area",
        "kitchen_area",
        "ceiling_height",
    ]
    outlier_cols = [c for c in outlier_cols if c in df.columns]

    for col in outlier_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df = df[(df[col] >= lower) & (df[col] <= upper)]

    return df


# Чтение данных
raw_df = pd.read_csv(DATA_PATH)

# Минимальная очистка
df = transform(raw_df)

# Таргет
target_col = "price"
if target_col not in df.columns:
    raise ValueError(f"В данных нет столбца '{target_col}'")

# Удаление строк без целевой переменной
df = df.dropna(subset=[target_col])

# Разделение на признаки и целевую переменную
X = df.drop(columns=[target_col])
y = df[target_col]

# Определение категориальных признаков для CatBoost
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
cat_features = [X.columns.get_loc(col) for col in cat_cols]

print(f"Форма данных после очистки: X={X.shape}, y={y.shape}")
print(f"Категориальные признаки: {cat_cols}")

# Разделение на train / valid / test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42
)
# В итоге: train 60%, valid 20%, test 20%

# Базовая модель CatBoostRegressor
model = CatBoostRegressor(
    loss_function="RMSE",
    depth=6,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42,
    verbose=False,
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_features if cat_features else None,
    use_best_model=True,
)


def print_regression_metrics(y_true, y_pred, prefix: str = "") -> None:
    """Печатает базовые метрики регрессии."""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{prefix}MSE: {mse:.2f}")
    print(f"{prefix}MAE: {mae:.2f}")
    print(f"{prefix}R2: {r2:.4f}")


# Метрики на валидации
y_valid_pred = model.predict(X_valid)
print("Метрики на валидации:")
print_regression_metrics(y_valid, y_valid_pred, prefix="valid_")

# Метрики на тесте
y_test_pred = model.predict(X_test)
print("\nМетрики на тесте:")
print_regression_metrics(y_test, y_test_pred, prefix="test_")

# Сохранение модели
models_dir = Path("services/models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "catboost_price_model.bin"
model.save_model(model_path)

print(f"\nМодель сохранена в: {model_path}")


/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


Форма данных после очистки: X=(64393, 20), y=(64393,)
Категориальные признаки: ['floors_total_bin', 'floor_bin', 'rooms_bin', 'build_year_bin']
Метрики на валидации:
valid_MSE: 3953538670110.66
valid_MAE: 1515045.98
valid_R2: 0.6937

Метрики на тесте:
test_MSE: 3973239865246.78
test_MAE: 1515604.73
test_R2: 0.6977

Модель сохранена в: services/models/catboost_price_model.bin


In [17]:
df.iloc[0].to_dict()

{'building_id': 6220,
 'build_year': nan,
 'building_type_int': 6,
 'latitude': 55.71711349487305,
 'longitude': 37.78112030029297,
 'ceiling_height': 2.640000104904175,
 'flats_count': 84,
 'floors_total': nan,
 'has_elevator': 1,
 'flat_id': 0,
 'floor': nan,
 'kitchen_area': 9.899999618530272,
 'living_area': 19.89999961853028,
 'rooms': nan,
 'is_apartment': 0,
 'total_area': 35.099998474121094,
 'floors_total_bin': 'low',
 'floor_bin': 'low',
 'rooms_bin': 'low',
 'build_year_bin': 'med',
 'price': 9500000.0}

In [ ]:
MODEL_PATH = "/home/mle-user/mle-project-sprint-1-v001/part2_dvc/models/fitted_model.pkl"

sys.path.append("/home/mle-user/mle-project-sprint-1-v001/part2_dvc/scripts")

import func

model = joblib.load(MODEL_PATH)
type(model)


/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.4.1.post1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.1.post1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from ver

sklearn.pipeline.Pipeline

In [ ]:

metrics = {}

y_pred_valid = model.predict(X_valid)
y_pred_test = model.predict(X_test)

# VALID
mse_valid = mean_squared_error(y_valid, y_pred_valid)
metrics["rmse_valid"] = np.sqrt(mse_valid)
metrics["mae_valid"] = mean_absolute_error(y_valid, y_pred_valid)
metrics["r2_valid"] = r2_score(y_valid, y_pred_valid)

# TEST
mse_test = mean_squared_error(y_test, y_pred_test)
metrics["rmse_test"] = np.sqrt(mse_test)
metrics["mae_test"] = mean_absolute_error(y_test, y_pred_test)
metrics["r2_test"] = r2_score(y_test, y_pred_test)

metrics


{'rmse_valid': 3853094.857278661,
 'mae_valid': 2593370.109375714,
 'r2_valid': 0.638142553480225,
 'rmse_test': 4009894.2234614263,
 'mae_test': 2612591.627495309,
 'r2_test': 0.6153287190876718}

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv(dotenv_path="../.env")

mlflow_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL")
aws_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret = os.getenv("AWS_SECRET_ACCESS_KEY")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = mlflow_s3_endpoint
os.environ["AWS_ACCESS_KEY_ID"] = aws_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret


In [ ]:
EXPERIMENT_NAME = "real_estate_price"
RUN_NAME = "baseline_train_valid_test"
REGISTRY_MODEL_NAME = "real_estate_price_model"

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

signature = infer_signature(X_test, y_pred_test)
input_example = X_test.iloc[:5]

metadata = {
    "stage": "baseline",
    "split_scheme": "train/valid/test = 60/20/20",
    "source_project": "mle-project-sprint-1-v001/part2_dvc",
    "model_type": type(model.named_steps['model']).__name__,
}

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id

    mlflow.log_metrics(metrics)

    if hasattr(model.named_steps["model"], "get_params"):
        mlflow.log_params(model.named_steps["model"].get_params())

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements="requirements.txt",
        metadata=metadata,
        await_registration_for=60,
    )

run_id, model_info.model_uri


/home/mle-user/.venv_mlflow/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'real_estate_price_model'.
2025/11/30 21:07:45 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to 

('0858c9bf45b843a48af89049e9f93aef',
 'runs:/0858c9bf45b843a48af89049e9f93aef/model')